In [1]:
from langchain_core.documents import Document

In [2]:
%pip install langchain langchain-core langchain-community

Note: you may need to restart the kernel to use updated packages.


In [3]:
doc = Document(page_content = "hello world",
               metadata = {
                   "source":"example.txt",
                   "pages":1,
                   "author":"Krish Naik",
                   "date_created":"2026-02-09"
               })
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2026-02-09'}, page_content='hello world')

In [4]:
!pip install langchain_community


Defaulting to user installation because normal site-packages is not writeable
^C


In [2]:
from langchain_community.document_loaders import TextLoader


/projectnb/cs585/students/siddhank/torch_env/lib/python3.10/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from pathlib import Path

BASE_DIR = Path().resolve()
print(BASE_DIR)

/projectnb/cs585/students/siddhank/data_research


In [4]:

file_path = BASE_DIR/ "data" / "test.txt"
loader = TextLoader(str(file_path), encoding="utf-8")
document=loader.load()


In [5]:
from langchain_community.document_loaders import DirectoryLoader

In [6]:
file_path=BASE_DIR / "data"
dir_loader=DirectoryLoader(str(file_path),
    glob="**/*.txt", ## Pattern to match files
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()


In [7]:
type(documents[0])

langchain_core.documents.base.Document

Embeddings and VectordB

In [11]:
!pip install sentence_transformers faiss-cpu chromadb

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu chromadb

In [7]:
import sentence_transformers
print(sentence_transformers.__version__)

5.2.3


In [14]:
import sys
!{sys.executable} -m pip install -U pysqlite3-binary

In [8]:
import sys, pysqlite3
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
import sqlite3
print(sqlite3.sqlite_version)
import chromadb

3.51.1


In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [11]:
class VectorStore:
    """
    In-memory ChromaDB vector store (Google Colab compatible)
    Stores embeddings created from your Drive text files
    """
    def __init__(self, collection_name: str = "txt_documents"):
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        try:
            # Ephemeral in-memory database
            self.client = chromadb.Client()

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"}  # better similarity search
            )

            print("Vector store ready")
            print("Collection:", self.collection_name)
            print("Existing documents:", self.collection.count())

        except Exception as e:
            print("Failed to initialize ChromaDB:", e)
            raise

    # -------------------------------------------------------
    # Add chunked documents + embeddings
    # -------------------------------------------------------
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings count mismatch")

        ids = []
        metadatas = []
        texts = []
        embedding_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):

            # unique id
            doc_id = f"doc_{uuid.uuid4().hex[:10]}"
            ids.append(doc_id)

            # metadata
            metadata = dict(doc.metadata)
            metadata["chunk_id"] = i
            metadata["length"] = len(doc.page_content)
            metadatas.append(metadata)

            # text
            texts.append(doc.page_content)

            # embedding
            embedding_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=texts
            )

            print(f"Added {len(texts)} chunks to vector database")
            print("Total stored:", self.collection.count())
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store ready
Collection: txt_documents
Existing documents: 0


In [19]:
!pip install --upgrade langchain

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [20]:
!pip install -U langchain langchain-community langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for langchain-text-splitters from https://files.pythonhosted.org/packages/84/66/d9e0c3b83b0ad75ee746c51ba347cacecb8d656b96e1d513f3e334d1ccab/langchain_text_splitters-1.1.1-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-core<2.0.0,>=1.2.10 from https://files.pythonhosted.org/packages/48/e0/a6a83dde94400b43d9b091ecbb41a50d6f86c4fecacb81b13d8452a7712b/langchain_core-1.2.15-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.12
    Uninstalling langchain-core-1.2.12:
      Successfully uninstalled langchain-core-1.2.12
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.0
    Uninstalling langchain-text-splitters-1.1.0:
      Successfully uninstalled

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
def split_documents(documents,chunk_size=500,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [14]:
chunks=split_documents(documents)


Split 3 documents into 100232 chunks

Example chunk:
Content: Beginners BBQ Class Taking Place in Missoula! Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class...
Metadata: {'source': '/projectnb/cs585/students/siddhank/data_research/data/train.txt'}


In [15]:
texts=[doc.page_content for doc in chunks]



embeddings=embedding_manager.generate_embeddings(texts)


def add_documents_in_batches(vectorstore, chunks, embeddings, batch_size=5000):
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch_chunks = chunks[i:i+batch_size]
        batch_embeddings = embeddings[i:i+batch_size]
        vectorstore.add_documents(batch_chunks, batch_embeddings)
        print(f"Added batch {i} → {i+len(batch_chunks)}")

Generating embeddings for 100232 texts...


Batches:   0%|          | 0/3133 [00:00<?, ?it/s]

Generated embeddings with shape: (100232, 384)


In [16]:
add_documents_in_batches(vectorstore, chunks, embeddings, batch_size=5000)

Added 5000 chunks to vector database
Total stored: 5000
Added batch 0 → 5000
Added 5000 chunks to vector database
Total stored: 10000
Added batch 5000 → 10000
Added 5000 chunks to vector database
Total stored: 15000
Added batch 10000 → 15000
Added 5000 chunks to vector database
Total stored: 20000
Added batch 15000 → 20000
Added 5000 chunks to vector database
Total stored: 25000
Added batch 20000 → 25000
Added 5000 chunks to vector database
Total stored: 30000
Added batch 25000 → 30000
Added 5000 chunks to vector database
Total stored: 35000
Added batch 30000 → 35000
Added 5000 chunks to vector database
Total stored: 40000
Added batch 35000 → 40000
Added 5000 chunks to vector database
Total stored: 45000
Added batch 40000 → 45000
Added 5000 chunks to vector database
Total stored: 50000
Added batch 45000 → 50000
Added 5000 chunks to vector database
Total stored: 55000
Added batch 50000 → 55000
Added 5000 chunks to vector database
Total stored: 60000
Added batch 55000 → 60000
Added 5000 

In [19]:
pip install pynvml

Note: you may need to restart the kernel to use updated packages.


In [17]:
from pynvml import (
    nvmlInit, nvmlShutdown, nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetUtilizationRates, nvmlDeviceGetMemoryInfo
)

def get_gpu_utilization():
    """
    Returns (gpu_util_percent, mem_util_percent)
    If no GPU or NVML not available -> (None, None)
    """
    try:
        nvmlInit()
        h = nvmlDeviceGetHandleByIndex(0)
        util = nvmlDeviceGetUtilizationRates(h)
        mem = nvmlDeviceGetMemoryInfo(h)
        gpu_util = float(util.gpu)   # %
        mem_util = (mem.used / mem.total) * 100
        nvmlShutdown()
        return gpu_util, float(mem_util)
    except Exception:
        return None, None

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [19]:
rag_retriever

In [28]:
rag_retriever.retrieve("What is python programming language?")

Retrieving documents for query: 'What is python programming language?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8c08e1b158',
  'content': 'Beginning Python: From Novice to Professional is the most comprehensive book on the Python ever written. Based on Practical Python, this newly-revised book is both an introduction and practical reference for a swath of Python-related programming topics, including addressing language internals, database integration, network programming, and web services. Advanced topics, such as extending Python and packaging/distributing Python applications, are also covered. Ten different projects illustrate',
  'metadata': {'source': '/projectnb/cs585/students/siddhank/data_research/data/train.txt',
   'length': 497,
   'chunk_id': 1759},
  'similarity_score': 0.6427603960037231,
  'distance': 0.35723960399627686,
  'rank': 1},
 {'id': 'doc_a9cf03d6ee',
  'content': 'integration, network programming, and web services. Advanced topics, such as extending Python and packaging/distributing Python applications, are also covered. Ten different projects illustrate the

In [35]:
import sys
!{sys.executable} -m pip install -U transformers accelerate huggingface_hub torch ipywidgets

In [36]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [27]:
from huggingface_hub import login


In [28]:
login()

In [39]:
from langchain_community.llms import LlamaCpp # Local LLaMA models


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

class LLMManager:
    def __init__(
        self,
        model_name: str,
        device_map: str = "auto",
        dtype: str = "auto",          # "auto", "float16", "bfloat16", "float32"
        load_in_4bit: bool = False,
        load_in_8bit: bool = False,
        max_new_tokens: int = 160,
        temperature: float = 0.5,
        do_sample: bool = True,
        trust_remote_code: bool = True,
    ):
        self.model_name = model_name
        self.device_map = device_map
        self.dtype = dtype
        self.load_in_4bit = load_in_4bit
        self.load_in_8bit = load_in_8bit
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.do_sample = do_sample
        self.trust_remote_code = trust_remote_code
        self.load_model()

    def _resolve_dtype(self):
        if self.dtype == "auto":
            # Prefer bf16 on Ampere+ if available; else fp16; else fp32.
            if torch.cuda.is_available():
                major = torch.cuda.get_device_properties(0).major
                return torch.bfloat16 if major >= 8 else torch.float16
            return torch.float32
        return {
            "float16": torch.float16,
            "bfloat16": torch.bfloat16,
            "float32": torch.float32,
        }[self.dtype]

    def load_model(self):
        print(f"\nLoading model: {self.model_name}\n")

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=self.trust_remote_code)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        torch_dtype = self._resolve_dtype()

        quant_config = None
        if self.load_in_4bit or self.load_in_8bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=self.load_in_4bit,
                load_in_8bit=self.load_in_8bit,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16 if torch_dtype == torch.bfloat16 else torch_dtype,
            )

        # If quantizing, dtype is mostly handled by quantization_config
        model_kwargs = dict(
            low_cpu_mem_usage=True,
            trust_remote_code=self.trust_remote_code,
            device_map=self.device_map if torch.cuda.is_available() else None,
        )
        if quant_config is not None:
            model_kwargs["quantization_config"] = quant_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **model_kwargs)

        # IMPORTANT: with device_map="auto", do NOT pass device=... to pipeline
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            do_sample=self.do_sample,
            return_full_text=False,
        )

        self.is_chat_model = hasattr(self.tokenizer, "chat_template") and self.tokenizer.chat_template is not None
        print("Detected CHAT model" if self.is_chat_model else "Detected COMPLETION model")
        print("Model ready!\n")

    def switch_model(
        self,
        new_model: str,
        *,
        dtype: str = None,
        load_in_4bit: bool = None,
        load_in_8bit: bool = None,
        device_map: str = None,
    ):
        print("\nSwitching model...\n")
        self.model_name = new_model
        if dtype is not None:
            self.dtype = dtype
        if load_in_4bit is not None:
            self.load_in_4bit = load_in_4bit
        if load_in_8bit is not None:
            self.load_in_8bit = load_in_8bit
        if device_map is not None:
            self.device_map = device_map
        self.load_model()

    def invoke(self, prompt: str) -> str:
        if self.is_chat_model:
            messages = [
                {"role": "system", "content": "Answer the question using the provided context only."},
                {"role": "user", "content": prompt},
            ]
            formatted_prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            return self.pipe(formatted_prompt)[0]["generated_text"].strip()
        else:
            return self.pipe(prompt)[0]["generated_text"].strip()

In [18]:
def rag_simple(query, retriever, llm, top_k=3):

    # retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""You are a helpful assistant.
Use ONLY the provided context to answer.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response


In [19]:
llm_manager = LLMManager("TinyLlama/TinyLlama-1.1B-Chat-v1.0")


Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Detected CHAT model
Model ready!



In [44]:
llm_manager.switch_model("Qwen/Qwen1.5-0.5B-Chat")


Switching model...


Loading model: Qwen/Qwen1.5-0.5B-Chat



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Detected CHAT model
Model ready!



In [46]:
llm_manager.switch_model(
    "Qwen/Qwen1.5-7B-Chat",
    dtype="float16",
    load_in_4bit=False,
    device_map="auto"
)


Switching model...


Loading model: Qwen/Qwen1.5-7B-Chat



Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

Detected CHAT model
Model ready!



In [20]:
llm_manager.switch_model(
    "meta-llama/Llama-3.3-70B-Instruct",
    dtype="bfloat16",
    load_in_4bit=False,
    device_map="auto"
)


Switching model...


Loading model: meta-llama/Llama-3.3-70B-Instruct



Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Detected CHAT model
Model ready!



In [27]:
from huggingface_hub import logout
logout()

In [23]:
from huggingface_hub import whoami
whoami()

{'type': 'user',
 'id': '68caf5ddf3b0f10cfcafd025',
 'name': 'siddhanthagenticAI',
 'fullname': 'Siddhanth Kalyanaraman',
 'email': 'siddhank@bu.edu',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1772323200,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/P6Kh9tm1ElYbUHw87QlIH.png',
 'orgs': [{'type': 'org',
   'id': '638908495a3d2a335625c2fa',
   'name': 'spark-ds549',
   'fullname': 'BU Spark! DS-549',
   'email': 'sparksub@bu.edu',
   'canPay': False,
   'billingMode': 'prepaid',
   'periodEnd': 1772323200,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/1669941978376-633b7c1f976a7d6910ef7a62.png',
   'roleInOrg': 'write',
   'isEnterprise': False},
  {'type': 'org',
   'id': '681b0cb0dba891d54be0773d',
   'name': 'mcp-course',
   'fullname': 'Hugging Face MCP Course',
   'email': None,
   'canPay': False,
   'billingMode': 'postpaid',
   'periodEnd': None,
   'avatarUrl'

In [22]:
from huggingface_hub import login
login()

In [49]:
llm_manager.switch_model("meta-llama/Llama-2-7b-chat-hf")


Switching model...


Loading model: meta-llama/Llama-2-7b-chat-hf



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Detected CHAT model
Model ready!



In [ ]:
llm_manager.switch_model("distilbert/distilgpt2")

In [50]:
prompt = """You are a helpful assistant.
Use ONLY the provided context to answer.

Context:
Machines are tools that help humans perform tasks more efficiently.

Question: What is a machine?

Answer:"""
answer = llm_manager.invoke(prompt)
print(answer)

Both `max_new_tokens` (=160) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the provided context, a machine is a tool that helps humans perform tasks more efficiently.


In [55]:


# Use your existing RAG retriever
query = "About Background Briefing?"
answer = rag_simple(query, rag_retriever, llm_manager)
print(answer)

Retrieving documents for query: 'About Background Briefing?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: module 'chromadb' has no attribute 'get_settings'


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Background Briefing on ABC Radio National was about disasters, including planning, management, and communication.


In [ ]:
print(answer)

In [24]:
answer=rag_simple("What is a machine?",rag_retriever,llm_manager)
print(answer)

Retrieving documents for query: 'What is a machine?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The context does not provide a direct definition of what a machine is. However, based on the provided text, it can be inferred that a machine can refer to various devices or equipment, such as a laptop, a grinding mill, a coffee machine (e.g. Rocket Espresso machine), or a machine tool used in a facility for manufacturing or production.


In [ ]:
print(answer)

In [26]:
def _to_text(x):
    """Best-effort conversion of various LLM outputs to plain text."""
    if x is None:
        return ""
    if isinstance(x, str):
        return x

    # dict outputs (common in custom managers)
    if isinstance(x, dict):
        for key in ("answer", "output", "text", "content", "generated_text", "response"):
            if key in x and isinstance(x[key], str):
                return x[key]
        # last resort: stringify
        return str(x)

    # LangChain-style AIMessage / HumanMessage etc.
    if hasattr(x, "content") and isinstance(getattr(x, "content"), str):
        return x.content

    # list outputs (some pipelines return list of generations)
    if isinstance(x, list) and x:
        return _to_text(x[0])

    return str(x)


def _call_llm(llm, prompt: str) -> str:
    """Call LLMManager / HF pipeline / LangChain LLM in a compatible way."""
    # 1) LangChain-style
    if hasattr(llm, "invoke"):
        return _to_text(llm.invoke(prompt))

    # 2) Your custom manager may expose generate()
    if hasattr(llm, "generate"):
        return _to_text(llm.generate(prompt))

    # 3) Some wrappers are callable
    if callable(llm):
        return _to_text(llm(prompt))

    raise TypeError(
        "llm must provide .invoke(prompt), .generate(prompt), or be callable."
    )


def rag_advanced(query, retriever, llm, top_k, min_score=0.2, return_context=False):
    # Retrieve documents
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        out = {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
        }
        if return_context:
            out["context"] = ""
        return out

    # Build context
    context = "\n\n".join(doc.get("content", "") for doc in results)

    # Sources
    sources = []
    for doc in results:
        md = doc.get("metadata", {}) or {}
        content = doc.get("content", "") or ""
        sources.append({
            "source": md.get("source_file", md.get("source", "unknown")),
            "page": md.get("page", "unknown"),
            "score": float(doc.get("similarity_score", 0.0)),
            "preview": (content[:300] + "...") if len(content) > 300 else content
        })

    confidence = max(float(doc.get("similarity_score", 0.0)) for doc in results)

    prompt = f"""You are a question answering assistant.

Answer ONLY using the provided context.
If the answer is not contained in the context, say: "I don't know."

Context:
{context}

Question: {query}

Answer:
"""

    response_text = _call_llm(llm, prompt).strip()

    out = {
        "answer": response_text,
        "sources": sources,
        "confidence": confidence,
    }
    if return_context:
        out["context"] = context
    return out

In [ ]:
result = rag_advanced(
    "Background Briefing",
    rag_retriever,
    llm_manager,
    top_k=2,
    min_score=0.1,
    return_context=True
)

print("Answer:", result["answer"])
print("Confidence:", result["confidence"])

In [27]:
import re
import time
import math
import numpy as np
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Any

# --- helpers you already effectively have ---
def _to_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, dict):
        for k in ("answer", "output", "text", "content", "generated_text", "response"):
            if k in x and isinstance(x[k], str):
                return x[k]
        return str(x)
    if hasattr(x, "content") and isinstance(getattr(x, "content"), str):
        return x.content
    if isinstance(x, list) and x:
        return _to_text(x[0])
    return str(x)

def _call_llm(llm, prompt: str) -> str:
    if hasattr(llm, "invoke"):
        return _to_text(llm.invoke(prompt))
    if hasattr(llm, "generate"):
        return _to_text(llm.generate(prompt))
    if callable(llm):
        return _to_text(llm(prompt))
    raise TypeError("llm must provide .invoke(prompt), .generate(prompt), or be callable.")

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    # simple sentence splitter
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

def contains_refusal(ans: str) -> bool:
    a = (ans or "").lower()
    return ("i don't know" in a) or ("i do not know" in a) or ("not in the context" in a) or ("no relevant context" in a)

In [28]:
import json

GROUNDING_PROMPT = """You are evaluating a RAG system.

Given:
- Context: passages retrieved from a knowledge base
- Claim: a single sentence from the model answer

Task:
Decide if the claim is FULLY supported by the Context.
- "supported": claim is directly stated or unambiguously implied by context
- "partially_supported": some support but missing key specifics / not fully justified
- "unsupported": not supported or contradicted by context

Return ONLY valid JSON with keys:
{{
  "label": "supported|partially_supported|unsupported",
  "evidence": "short quote or phrase from context (or empty if unsupported)"
}}

Context:
{context}

Claim:
{claim}
"""

QA_RELEVANCE_PROMPT = """You are evaluating a RAG system.

Given:
- Question
- Answer

Score how well the Answer addresses the Question, regardless of grounding.
Return ONLY valid JSON:
{{
  "score": <integer 1-5>,
  "reason": "<short reason>"
}}

Question:
{question}

Answer:
{answer}
"""

CONTEXT_RELEVANCE_PROMPT = """You are evaluating a RAG system.

Given:
- Question
- Context

Score how relevant the Context is to answering the Question.
Return ONLY valid JSON:
{{
  "score": <integer 1-5>,
  "reason": "<short reason>"
}}

Question:
{question}

Context:
{context}
"""

In [29]:
def safe_json_loads(s: str) -> Dict[str, Any]:
    s = (s or "").strip()
    # try direct
    try:
        return json.loads(s)
    except Exception:
        pass
    # try extracting first json object
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}
    return {}

In [30]:
LABEL_TO_SCORE = {
    "supported": 1.0,
    "partially_supported": 0.5,
    "unsupported": 0.0
}

def judge_claims(llm_judge, context: str, answer: str, max_claims: int = 12) -> Dict[str, Any]:
    claims = split_sentences(answer)[:max_claims]
    judged = []
    for c in claims:
        p = GROUNDING_PROMPT.format(context=context, claim=c)
        raw = _call_llm(llm_judge, p)
        obj = safe_json_loads(raw)
        label = (obj.get("label") or "").strip().lower()
        if label not in LABEL_TO_SCORE:
            # fallback: treat as unsupported if malformed
            label = "unsupported"
            obj["evidence"] = ""
        judged.append({
            "claim": c,
            "label": label,
            "evidence": obj.get("evidence", "")
        })

    if not judged:
        return {
            "claims": [],
            "hallucination_rate": 0.0,
            "groundedness_score": 0.0,
            "unsupported_count": 0,
            "claim_count": 0
        }

    scores = [LABEL_TO_SCORE[j["label"]] for j in judged]
    unsupported = sum(1 for j in judged if j["label"] == "unsupported")
    claim_count = len(judged)
    halluc_rate = unsupported / claim_count
    grounded_score = float(np.mean(scores))

    return {
        "claims": judged,
        "hallucination_rate": float(halluc_rate),
        "groundedness_score": grounded_score,
        "unsupported_count": int(unsupported),
        "claim_count": int(claim_count)
    }

def judge_relevance(llm_judge, question: str, answer: str, context: str) -> Dict[str, Any]:
    qa_raw = _call_llm(llm_judge, QA_RELEVANCE_PROMPT.format(question=question, answer=answer))
    qa_obj = safe_json_loads(qa_raw)
    qa_score = int(qa_obj.get("score", 0) or 0)

    ctx_raw = _call_llm(llm_judge, CONTEXT_RELEVANCE_PROMPT.format(question=question, context=context))
    ctx_obj = safe_json_loads(ctx_raw)
    ctx_score = int(ctx_obj.get("score", 0) or 0)

    return {
        "answer_relevance_1to5": qa_score,
        "answer_relevance_reason": qa_obj.get("reason", ""),
        "context_relevance_1to5": ctx_score,
        "context_relevance_reason": ctx_obj.get("reason", "")
    }

In [31]:
@dataclass
class RagEvalRow:
    query: str
    answer: str
    confidence: float
    latency_s: float
    hallucination_rate: float
    groundedness_score: float
    answer_relevance_1to5: int
    context_relevance_1to5: int
    refusal: bool
    top_source: str

def evaluate_rag(
    queries: List[Dict[str, Any]],
    retriever,
    rag_llm,
    judge_llm,
    top_k: int = 3,
    min_score: float = 0.1,
    max_claims: int = 12
):
    rows: List[RagEvalRow] = []
    detailed: List[Dict[str, Any]] = []

    for item in queries:
        q = item["query"]
        t0 = time.time()
        result = rag_advanced(q, retriever, rag_llm, top_k=top_k, min_score=min_score, return_context=True)
        latency = time.time() - t0

        answer = result.get("answer", "")
        context = result.get("context", "")
        confidence = float(result.get("confidence", 0.0))
        sources = result.get("sources", []) or []
        top_source = sources[0]["source"] if sources else ""

        # claim-level grounding metrics
        grounding = judge_claims(judge_llm, context=context, answer=answer, max_claims=max_claims)

        # relevance metrics
        rel = judge_relevance(judge_llm, question=q, answer=answer, context=context)

        refusal = contains_refusal(answer)

        row = RagEvalRow(
            query=q,
            answer=answer,
            confidence=confidence,
            latency_s=float(latency),
            hallucination_rate=grounding["hallucination_rate"],
            groundedness_score=grounding["groundedness_score"],
            answer_relevance_1to5=int(rel["answer_relevance_1to5"] or 0),
            context_relevance_1to5=int(rel["context_relevance_1to5"] or 0),
            refusal=bool(refusal),
            top_source=top_source
        )
        rows.append(row)

        detailed.append({
            "query": q,
            "result": result,
            "grounding": grounding,
            "relevance": rel
        })

    return rows, detailed

In [32]:
def summarize_eval(rows: List[RagEvalRow]) -> Dict[str, Any]:
    if not rows:
        return {}

    hall = [r.hallucination_rate for r in rows]
    ground = [r.groundedness_score for r in rows]
    ansrel = [r.answer_relevance_1to5 for r in rows]
    ctxrel = [r.context_relevance_1to5 for r in rows]
    lat = [r.latency_s for r in rows]
    conf = [r.confidence for r in rows]
    refusals = sum(1 for r in rows if r.refusal)

    return {
        "n": len(rows),
        "avg_hallucination_rate": float(np.mean(hall)),
        "avg_groundedness_score": float(np.mean(ground)),
        "avg_answer_relevance_1to5": float(np.mean(ansrel)),
        "avg_context_relevance_1to5": float(np.mean(ctxrel)),
        "avg_latency_s": float(np.mean(lat)),
        "avg_confidence": float(np.mean(conf)),
        "refusal_rate": refusals / len(rows)
    }

In [33]:
test_queries = [
    {"query": "Who is the CEO of the company ABC Radio?"},
    {"query": "What is the author's phone number?"},
    {"query": "Summarize future plans for 2030."}
]

rows, detailed = evaluate_rag(
    queries=test_queries,
    retriever=rag_retriever,
    rag_llm=llm_manager,     # the generator
    judge_llm=llm_manager,   # start with same model as judge
    top_k=2,
    min_score=0.1,
    max_claims=10
)

summary = summarize_eval(rows)
summary

Retrieving documents for query: 'Who is the CEO of the company ABC Radio?'
Top K: 2, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [40]:
import pandas as pd
df = pd.DataFrame([asdict(r) for r in rows])
df[["query","hallucination_rate","groundedness_score","answer_relevance_1to5","context_relevance_1to5","latency_s","confidence","refusal"]]

,query,hallucination_rate,groundedness_score,answer_relevance_1to5,context_relevance_1to5,latency_s,confidence,refusal
0,Who is the CEO of the company ABC Radio?,1.0,0.0,1,1,143.009292,0.516549,True
1,What is the author's phone number?,1.0,0.0,3,2,68.007523,0.505188,True
2,Summarize future plans for 2030.,1.0,0.0,1,1,68.309045,0.491593,True


In [61]:
llm_manager.switch_model("Qwen/Qwen1.5-7B-Chat")


Switching model...


Loading model: Qwen/Qwen1.5-7B-Chat



config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Detected CHAT model
Model ready!



In [34]:
test_queries = [
    {"query": "Who is the CEO of the company ABC Radio?"},
    {"query": "What is a machine?"},
    {"query": "Summarize key dates mentioned?"}
]

rows, detailed = evaluate_rag(
    queries=test_queries,
    retriever=rag_retriever,
    rag_llm=llm_manager,     # the generator
    judge_llm=llm_manager,   # start with same model as judge
    top_k=2,
    min_score=0.1,
    max_claims=10
)

summary = summarize_eval(rows)
summary

Retrieving documents for query: 'Who is the CEO of the company ABC Radio?'
Top K: 2, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati

{'n': 3,
 'avg_hallucination_rate': 0.7333333333333334,
 'avg_groundedness_score': 0.2333333333333333,
 'avg_answer_relevance_1to5': 2.3333333333333335,
 'avg_context_relevance_1to5': 2.6666666666666665,
 'avg_latency_s': 345.4208992322286,
 'avg_confidence': 0.5166328152020773,
 'refusal_rate': 0.6666666666666666}

In [35]:
import pandas as pd
from dataclasses import asdict

df = pd.DataFrame([asdict(r) for r in rows])
df[["query", "answer", "hallucination_rate", "groundedness_score", "answer_relevance_1to5", "context_relevance_1to5", "confidence", "latency_s", "top_source"]]

,query,answer,hallucination_rate,groundedness_score,answer_relevance_1to5,context_relevance_1to5,confidence,latency_s,top_source
0,Who is the CEO of the company ABC Radio?,I don't know.,1.0,0.0,1,1,0.516549,411.005962,/projectnb/cs585/students/siddhank/data_resear...
1,What is a machine?,I don't know.,1.0,0.0,1,2,0.538377,71.998778,/projectnb/cs585/students/siddhank/data_resear...
2,Summarize key dates mentioned?,The key dates mentioned are: \n1. 31 January 2...,0.2,0.7,5,5,0.494972,553.257958,/projectnb/cs585/students/siddhank/data_resear...


In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

NVIDIA L40S
44.39215087890625 GB


In [20]:
# =========================
# FULL CODE (NO __main__)
# Runs your test_queries, evaluates, and stores results in a pandas DataFrame `df`
# =========================

import re
import time
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig


# -------------------------
# LLM MANAGER (HF Pipeline)
# -------------------------
class LLMManager:
    def __init__(
        self,
        model_name: str,
        device_map: str = "auto",
        dtype: str = "auto",          # "auto", "float16", "bfloat16", "float32"
        load_in_4bit: bool = False,
        load_in_8bit: bool = False,
        max_new_tokens: int = 160,
        temperature: float = 0.5,
        do_sample: bool = True,
        trust_remote_code: bool = True,
    ):
        self.model_name = model_name
        self.device_map = device_map
        self.dtype = dtype
        self.load_in_4bit = load_in_4bit
        self.load_in_8bit = load_in_8bit
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.do_sample = do_sample
        self.trust_remote_code = trust_remote_code
        self.load_model()

    def _resolve_dtype(self):
        if self.dtype == "auto":
            if torch.cuda.is_available():
                major = torch.cuda.get_device_properties(0).major
                return torch.bfloat16 if major >= 8 else torch.float16
            return torch.float32
        return {
            "float16": torch.float16,
            "bfloat16": torch.bfloat16,
            "float32": torch.float32,
        }[self.dtype]

    def load_model(self):
        print(f"\nLoading model: {self.model_name}\n")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, trust_remote_code=self.trust_remote_code
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        torch_dtype = self._resolve_dtype()

        quant_config = None
        if self.load_in_4bit or self.load_in_8bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=self.load_in_4bit,
                load_in_8bit=self.load_in_8bit,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16 if torch_dtype == torch.bfloat16 else torch_dtype,
            )

        model_kwargs = dict(
            low_cpu_mem_usage=True,
            trust_remote_code=self.trust_remote_code,
            device_map=self.device_map if torch.cuda.is_available() else None,
        )
        if quant_config is not None:
            model_kwargs["quantization_config"] = quant_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **model_kwargs)

        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            do_sample=self.do_sample,
            return_full_text=False,
        )

        self.is_chat_model = hasattr(self.tokenizer, "chat_template") and self.tokenizer.chat_template is not None
        print("Detected CHAT model" if self.is_chat_model else "Detected COMPLETION model")
        print("Model ready!\n")

    def _format_prompt(self, prompt: str) -> str:
        if self.is_chat_model:
            messages = [
                {"role": "system", "content": "Answer the question using the provided context only."},
                {"role": "user", "content": prompt},
            ]
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        return prompt

    def invoke_timed(self, prompt: str) -> Dict[str, Any]:
        formatted_prompt = self._format_prompt(prompt)

        # prompt tokens
        try:
            prompt_tokens = len(self.tokenizer.encode(formatted_prompt))
        except Exception:
            prompt_tokens = 0

        # accurate timing for GPU
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        out = self.pipe(formatted_prompt)[0]["generated_text"].strip()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        llm_latency_s = t1 - t0

        # output tokens
        try:
            output_tokens = len(self.tokenizer.encode(out))
        except Exception:
            output_tokens = 0

        tps = (output_tokens / llm_latency_s) if llm_latency_s > 0 else 0.0

        return {
            "text": out,
            "llm_latency_s": float(llm_latency_s),
            "prompt_tokens": int(prompt_tokens),
            "output_tokens": int(output_tokens),
            "total_tokens": int(prompt_tokens + output_tokens),
            "tokens_per_sec": float(tps),
        }

    def invoke(self, prompt: str) -> str:
        return self.invoke_timed(prompt)["text"]

In [21]:
llm_manager = LLMManager("TinyLlama/TinyLlama-1.1B-Chat-v1.0")



Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Detected CHAT model
Model ready!



In [36]:

llm_manager = LLMManager("meta-llama/Llama-3.3-70B-Instruct")


Loading model: meta-llama/Llama-3.3-70B-Instruct



Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Detected CHAT model
Model ready!



In [37]:
# -------------------------
# UTIL HELPERS
# -------------------------
def _to_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, dict):
        for k in ("answer", "output", "text", "content", "generated_text", "response"):
            if k in x and isinstance(x[k], str):
                return x[k]
        return str(x)
    if hasattr(x, "content") and isinstance(getattr(x, "content"), str):
        return x.content
    if isinstance(x, list) and x:
        return _to_text(x[0])
    return str(x)

def _call_llm(llm, prompt: str) -> str:
    if hasattr(llm, "invoke"):
        return _to_text(llm.invoke(prompt))
    if hasattr(llm, "generate"):
        return _to_text(llm.generate(prompt))
    if callable(llm):
        return _to_text(llm(prompt))
    raise TypeError("llm must provide .invoke(prompt), .generate(prompt), or be callable.")

def _call_llm_timed(llm, prompt: str) -> Dict[str, Any]:
    if hasattr(llm, "invoke_timed"):
        return llm.invoke_timed(prompt)
    return {"text": _call_llm(llm, prompt), "llm_latency_s": None}

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", text)
    return [p.strip() for p in parts if p.strip()]

def contains_refusal(ans: str) -> bool:
    a = (ans or "").lower()
    return ("i don't know" in a) or ("i do not know" in a) or ("not in the context" in a) or ("no relevant context" in a)

def safe_json_loads(s: str) -> Dict[str, Any]:
    s = (s or "").strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}
    return {}

def now() -> float:
    return time.perf_counter()


# -------------------------
# RAG ADVANCED + TIMINGS
# -------------------------
def rag_advanced(query, retriever, llm, top_k, min_score=0.2, return_context=False, return_timings=True):
    timings = {}
    t_total0 = now()

    # Retrieve documents
    t0 = now()
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    timings["retrieve_s"] = now() - t0

    if not results:
        timings["total_s"] = now() - t_total0
        out = {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
        }
        if return_context:
            out["context"] = ""
        if return_timings:
            out["timings"] = timings
        return out

    # Build context
    t0 = now()
    context = "\n\n".join(doc.get("content", "") for doc in results)
    timings["prompt_build_s"] = now() - t0

    # Sources
    sources = []
    for doc in results:
        md = doc.get("metadata", {}) or {}
        content = doc.get("content", "") or ""
        sources.append({
            "source": md.get("source_file", md.get("source", "unknown")),
            "page": md.get("page", "unknown"),
            "score": float(doc.get("similarity_score", 0.0)),
            "preview": (content[:300] + "...") if len(content) > 300 else content
        })

    confidence = max(float(doc.get("similarity_score", 0.0)) for doc in results)

    prompt = f"""You are a question answering assistant.

Answer ONLY using the provided context.
If the answer is not contained in the context, say: "I don't know."

Context:
{context}

Question: {query}

Answer:
"""

    # LLM time (model-only generation)
    llm_out = _call_llm_timed(llm, prompt)
    response_text = (llm_out.get("text") or "").strip()

    timings["llm_s"] = float(llm_out.get("llm_latency_s") or 0.0)
    timings["output_tokens"] = llm_out.get("output_tokens")
    timings["tokens_per_sec"] = llm_out.get("tokens_per_sec")
    timings["total_s"] = now() - t_total0

    out = {
        "answer": response_text,
        "sources": sources,
        "confidence": confidence,
    }
    if return_context:
        out["context"] = context
    if return_timings:
        out["timings"] = timings
    return out


# -------------------------
# JUDGE PROMPTS
# -------------------------
GROUNDING_PROMPT = """You are evaluating a RAG system.

Given:
- Context: passages retrieved from a knowledge base
- Claim: a single sentence from the model answer

Task:
Decide if the claim is FULLY supported by the Context.
- "supported": claim is directly stated or unambiguously implied by context
- "partially_supported": some support but missing key specifics / not fully justified
- "unsupported": not supported or contradicted by context

Return ONLY valid JSON with keys:
{{
  "label": "supported|partially_supported|unsupported",
  "evidence": "short quote or phrase from context (or empty if unsupported)"
}}

Context:
{context}

Claim:
{claim}
"""
QA_RELEVANCE_PROMPT = """You are evaluating a RAG system.

Given:
- Question
- Answer

Score how well the Answer addresses the Question, regardless of grounding.
Return ONLY valid JSON:
{{
  "score": <integer 1-5>,
  "reason": "<short reason>"
}}

Question:
{question}

Answer:
{answer}
"""

CONTEXT_RELEVANCE_PROMPT = """You are evaluating a RAG system.

Given:
- Question
- Context

Score how relevant the Context is to answering the Question.
Return ONLY valid JSON:
{{
  "score": <integer 1-5>,
  "reason": "<short reason>"
}}

Question:
{question}

Context:
{context}
"""

LABEL_TO_SCORE = {"supported": 1.0, "partially_supported": 0.5, "unsupported": 0.0}

def judge_claims(llm_judge, context: str, answer: str, max_claims: int = 12) -> Dict[str, Any]:
    claims = split_sentences(answer)[:max_claims]
    judged = []

    for c in claims:
        p = GROUNDING_PROMPT.format(context=context, claim=c)
        raw = _call_llm(llm_judge, p)
        obj = safe_json_loads(raw)

        label = (obj.get("label") or "").strip().lower()
        if label not in LABEL_TO_SCORE:
            label = "unsupported"
            obj["evidence"] = ""

        judged.append({
            "claim": c,
            "label": label,
            "evidence": obj.get("evidence", "")
        })

    if not judged:
        return {
            "claims": [],
            "hallucination_rate": 0.0,
            "groundedness_score": 0.0,
            "unsupported_count": 0,
            "claim_count": 0
        }

    scores = [LABEL_TO_SCORE[j["label"]] for j in judged]
    unsupported = sum(1 for j in judged if j["label"] == "unsupported")
    claim_count = len(judged)

    return {
        "claims": judged,
        "hallucination_rate": float(unsupported / claim_count),
        "groundedness_score": float(np.mean(scores)),
        "unsupported_count": int(unsupported),
        "claim_count": int(claim_count)
    }

def judge_relevance(llm_judge, question: str, answer: str, context: str) -> Dict[str, Any]:
    qa_raw = _call_llm(llm_judge, QA_RELEVANCE_PROMPT.format(question=question, answer=answer))
    qa_obj = safe_json_loads(qa_raw)
    qa_score = int(qa_obj.get("score", 0) or 0)

    ctx_raw = _call_llm(llm_judge, CONTEXT_RELEVANCE_PROMPT.format(question=question, context=context))
    ctx_obj = safe_json_loads(ctx_raw)
    ctx_score = int(ctx_obj.get("score", 0) or 0)

    return {
        "answer_relevance_1to5": qa_score,
        "answer_relevance_reason": qa_obj.get("reason", ""),
        "context_relevance_1to5": ctx_score,
        "context_relevance_reason": ctx_obj.get("reason", ""),
    }


# -------------------------
# EVAL STRUCTURES (match your df columns request)
# NOTE: latency_s = TOTAL pipeline latency, like your original code
# -------------------------
@dataclass
class RagEvalRow:
    query: str
    answer: str
    hallucination_rate: float
    groundedness_score: float
    answer_relevance_1to5: int
    context_relevance_1to5: int
    confidence: float
    latency_s: float
    llm_latency_s: float
    retrieve_s: float
    tokens_per_sec: float
    gpu_util_percent: float
    gpu_mem_percent: float
    top_source: str
    refusal: bool

def evaluate_rag(
    queries: List[Dict[str, Any]],
    retriever,
    rag_llm,
    judge_llm,
    top_k: int = 3,
    min_score: float = 0.1,
    max_claims: int = 12
):
    rows: List[RagEvalRow] = []
    detailed: List[Dict[str, Any]] = []

    for item in queries:
        q = item["query"]

        # Run RAG (includes timings)
        result = rag_advanced(
            q, retriever, rag_llm,
            top_k=top_k, min_score=min_score,
            return_context=True, return_timings=True
        )

        answer = result.get("answer", "")
        context = result.get("context", "")
        confidence = float(result.get("confidence", 0.0))
        sources = result.get("sources", []) or []
        top_source = sources[0]["source"] if sources else ""
        gpu_util, gpu_mem = get_gpu_utilization()
        timings = result.get("timings", {}) or {}
        latency_s = float(timings.get("total_s", 0.0))
        llm_latency_s = float(timings.get("llm_s", 0.0))
        retrieve_s = float(timings.get("retrieve_s", 0.0))
        tokens_per_sec = float(timings.get("tokens_per_sec") or 0.0)

        # Grounding metrics
        grounding = judge_claims(judge_llm, context=context, answer=answer, max_claims=max_claims)

        # Relevance metrics
        rel = judge_relevance(judge_llm, question=q, answer=answer, context=context)

        refusal = contains_refusal(answer)

        row = RagEvalRow(
            query=q,
            answer=answer,
            hallucination_rate=float(grounding["hallucination_rate"]),
            groundedness_score=float(grounding["groundedness_score"]),
            answer_relevance_1to5=int(rel["answer_relevance_1to5"] or 0),
            context_relevance_1to5=int(rel["context_relevance_1to5"] or 0),
            confidence=float(confidence),
            latency_s=float(latency_s),
            llm_latency_s=float(llm_latency_s),
            retrieve_s=float(retrieve_s),
            tokens_per_sec=float(tokens_per_sec),
            gpu_util_percent=gpu_util,
            gpu_mem_percent=gpu_mem, 
            top_source=top_source,
            refusal=bool(refusal),
        )
        rows.append(row)

        detailed.append({
            "query": q,
            "result": result,
            "timings": timings,
            "grounding": grounding,
            "relevance": rel
        })

    return rows, detailed

def summarize_eval(rows: List[RagEvalRow]) -> Dict[str, Any]:
    if not rows:
        return {}
    return {
        "n": len(rows),
        "avg_hallucination_rate": float(np.mean([r.hallucination_rate for r in rows])),
        "avg_groundedness_score": float(np.mean([r.groundedness_score for r in rows])),
        "avg_answer_relevance_1to5": float(np.mean([r.answer_relevance_1to5 for r in rows])),
        "avg_context_relevance_1to5": float(np.mean([r.context_relevance_1to5 for r in rows])),
        "avg_latency_s": float(np.mean([r.latency_s for r in rows])),
        "avg_llm_latency_s": float(np.mean([r.llm_latency_s for r in rows])),
        "avg_retrieve_s": float(np.mean([r.retrieve_s for r in rows])),
        "avg_tokens_per_sec": float(np.mean([r.tokens_per_sec for r in rows])),
        "avg_confidence": float(np.mean([r.confidence for r in rows])),
        "refusal_rate": float(np.mean([1.0 if r.refusal else 0.0 for r in rows])),
    }

In [38]:
GROUNDING_PROMPT

'You are evaluating a RAG system.\n\nGiven:\n- Context: passages retrieved from a knowledge base\n- Claim: a single sentence from the model answer\n\nTask:\nDecide if the claim is FULLY supported by the Context.\n- "supported": claim is directly stated or unambiguously implied by context\n- "partially_supported": some support but missing key specifics / not fully justified\n- "unsupported": not supported or contradicted by context\n\nReturn ONLY valid JSON with keys:\n{{\n  "label": "supported|partially_supported|unsupported",\n  "evidence": "short quote or phrase from context (or empty if unsupported)"\n}}\n\nContext:\n{context}\n\nClaim:\n{claim}\n'

In [39]:
class SimpleDemoRetriever:
    def __init__(self, docs: List[Dict[str, Any]]):
        self.docs = docs

    def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.1):
        out = []
        for i, d in enumerate(self.docs[:top_k]):
            out.append({
                "content": d.get("content", ""),
                "metadata": d.get("metadata", {}),
                "similarity_score": float(d.get("similarity_score", 0.9 - 0.1*i)),
            })
        return [x for x in out if x["similarity_score"] >= score_threshold]


# =========================
# SETUP (edit these 2 lines)
# =========================
# 1) your model
# llm_manager = LLMManager("your-model-name-here")
#
# 2) your retriever
# rag_retriever = <your retriever>

# If you want a demo run without your retriever, uncomment these:
# llm_manager = LLMManager("gpt2", max_new_tokens=80)
# rag_retriever = SimpleDemoRetriever([
#     {"content": "ABC Radio is a fictional company. No CEO is given here.", "metadata": {"source": "demo"}, "similarity_score": 0.92},
#     {"content": "A machine is a device that uses power to apply forces and control movement.", "metadata": {"source": "demo"}, "similarity_score": 0.85},
#     {"content": "Key dates mentioned: January 2, 2020 and March 5, 2021.", "metadata": {"source": "demo"}, "similarity_score": 0.80},
# ])


# =========================
# TEST QUERIES + DF OUTPUT
# =========================
test_queries = [
    {"query": "Who is the CEO of the company ABC Radio?"},
    {"query": "What is a machine?"},
    {"query": "Summarize key dates mentioned?"}
]

rows, detailed = evaluate_rag(
    queries=test_queries,
    retriever=rag_retriever,
    rag_llm=llm_manager,     # the generator
    judge_llm=llm_manager,   # start with same model as judge
    top_k=2,
    min_score=0.1,
    max_claims=10
)

summary = summarize_eval(rows)

df = pd.DataFrame([asdict(r) for r in rows])

# your requested view:
df_view = df[[
    "query",
    "answer",
    "hallucination_rate",
    "groundedness_score",
    "answer_relevance_1to5",
    "context_relevance_1to5",
    "confidence",
    "latency_s",        # total RAG time
    "llm_latency_s",    # model-only time ✅
    "retrieve_s",       # retrieval time (optional but useful)
    "tokens_per_sec",   # generation speed (optional)
    "top_source",
    "gpu_util_percent",     # ✅
    "gpu_mem_percent",      # ✅
]]

# df_view is what you want to display in notebook
df_view

Retrieving documents for query: 'Who is the CEO of the company ABC Radio?'
Top K: 2, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,query,answer,hallucination_rate,groundedness_score,answer_relevance_1to5,context_relevance_1to5,confidence,latency_s,llm_latency_s,retrieve_s,tokens_per_sec,top_source,gpu_util_percent,gpu_mem_percent
0,Who is the CEO of the company ABC Radio?,I don't know.,1.0,0.0,1,1,0.516549,146.431803,146.405205,0.014508,0.040982,/projectnb/cs585/students/siddhank/data_resear...,94.0,97.314025
1,What is a machine?,I don't know.,1.0,0.0,1,2,0.538377,131.301681,130.956027,0.343671,0.045817,/projectnb/cs585/students/siddhank/data_resear...,100.0,97.314025
2,Summarize key dates mentioned?,The key dates mentioned are: \n1. 31 January 2...,0.2,0.7,5,5,0.494972,691.878166,691.860631,0.015535,0.062151,/projectnb/cs585/students/siddhank/data_resear...,0.0,1.325079
